# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis:** One row means one page, on one day, for one client. This comes 
from the table `fact_content_daily_performance`, where each row is identified by 
`report_date`, `client_hash_id`, and `content_hash_id` together.

**Table used:** `fact_content_daily_performance`

**Time window:** I am using March 2026 (`month=2026-03`) as my working month. This is 
a mid-panel month, not the last month, so I am not using the `_sample` table (which is 
only June 2026) to build my label logic.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Feature (safe to use):** `gsc_avg_position`, `gsc_impressions`, `gsc_clicks` — these 
are known at the time we check the page, before we make any decision about it.

**Label / proxy (what I am predicting):** My opportunity score, calculated as the gap 
between a page's expected CTR (based on similar-position pages) and its actual CTR 
(`gsc_clicks / gsc_impressions`). This is not a column that already exists — I 
calculate it myself, so it must never be used as an input feature.

**Context (for grouping/joining only, never as a feature):** `content_hash_id`, 
`client_hash_id`, `report_date` — these identify rows but do not describe page 
performance, so the model should never learn from them directly.

**Excluded (and why):** I am excluding rows where `gsc_data_available` is not TRUE, 
because those rows do not have real search data and would give false signals. I am 
also excluding any column that is directly derived from CTR itself, since that would 
leak the answer into the input.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [24]:
import os
from getpass import getpass

os.environ["HF_TOKEN"] = getpass("Paste your Hugging Face token: ")

In [25]:
import duckdb

con = duckdb.connect()
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

con.execute(f"""
    CREATE SECRET hf_token (
        TYPE HUGGINGFACE,
        TOKEN '{os.environ["HF_TOKEN"]}'
    );
""")

table_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

In [26]:
# Query 1: grain check (proves one row = one page-day)

grain_check = con.execute(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) as c
    FROM read_parquet('{table_path}')
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING c > 1
    LIMIT 5
""").df()

grain_check

,report_date,client_hash_id,content_hash_id,c


In [27]:
# Query 2: row count + date span

summary = con.execute(f"""
    SELECT COUNT(*) as row_count, MIN(report_date) as earliest, MAX(report_date) as latest
    FROM read_parquet('{table_path}')
""").df()

summary

,row_count,earliest,latest
0,9841378,2026-03-01,2026-03-31


In [28]:
# Query 3: availability check

availability = con.execute(f"""
    SELECT 
        COUNT(*) as total_rows,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) as gsc_available_rows
    FROM read_parquet('{table_path}')
""").df()

availability

,total_rows,gsc_available_rows
0,9841378,3611061.0


In [29]:
# 5 features

features = con.execute(f"""
    SELECT 
        content_hash_id,
        client_hash_id,
        report_date,
        gsc_avg_position,
        gsc_impressions,
        gsc_clicks,
        CASE WHEN gsc_impressions > 0 THEN gsc_clicks * 1.0 / gsc_impressions ELSE NULL END as actual_ctr
    FROM read_parquet('{table_path}')
    WHERE gsc_data_available IS TRUE
    LIMIT 1000
""").df()

features.head()

,content_hash_id,client_hash_id,report_date,gsc_avg_position,gsc_impressions,gsc_clicks,actual_ctr
0,content_b7e512995f79d5a6,client_73cda7b4e4f265ea,2026-03-01,3.350000,20,0,0.000
1,content_05597932fe4da067,client_73cda7b4e4f265ea,2026-03-01,0.000000,1,0,0.000
2,content_7a105f548d9c6916,client_73cda7b4e4f265ea,2026-03-01,4.928000,125,1,0.008
3,content_905aa32a0230694e,client_73cda7b4e4f265ea,2026-03-01,4.000000,7,0,0.000
4,content_a3ea9792f793ec72,client_73cda7b4e4f265ea,2026-03-01,2.272727,11,0,0.000



- `gsc_avg_position` — available because this is the page's search ranking position, 
  already recorded for that day in Search Console before any decision is made.
- `gsc_impressions` — available because this is how many times the page was shown in 
  search results that day, already logged by the time we look at the data.
- `gsc_clicks` — available because this is how many clicks the page got that day, 
  already recorded alongside impressions.
- `actual_ctr` — available because it is calculated directly from `gsc_clicks` and 
  `gsc_impressions`, both of which are already known for that day.
- `report_date` — available because this is just the day the data was collected, known 
  by definition before any analysis happens.

In [30]:
import pandas as pd

# real target: opportunity score = expected CTR (by position group) minus actual CTR
features["position_group"] = pd.cut(features["gsc_avg_position"], bins=[0, 3, 6, 10, 100])
features["opportunity_score"] = features.groupby("position_group")["actual_ctr"].transform("mean") - features["actual_ctr"]

# a fake feature that secretly copies the answer
features["leaky_column"] = features["opportunity_score"] * 10

# check correlation with honest features vs with the leaky one
print("Honest features only:")
print(features[["gsc_avg_position", "gsc_impressions"]].corrwith(features["opportunity_score"]))

print("\nWith the leaky column added:")
print(features[["gsc_avg_position", "gsc_impressions", "leaky_column"]].corrwith(features["opportunity_score"]))

# remove the trap
features = features.drop(columns=["leaky_column"])

Honest features only:
gsc_avg_position    0.033020
gsc_impressions    -0.009324
dtype: float64

With the leaky column added:
gsc_avg_position    0.033020
gsc_impressions    -0.009324
leaky_column        1.000000
dtype: float64


I added `leaky_column`, which was calculated directly from `opportunity_score` itself 
(just multiplied by 10). When I checked its correlation with the target, it came out 
to a perfect 1.0 — far higher than any honest feature like position or impressions. 
This is leakage: the "feature" was not real information, it was the answer in 
disguise. In a real model, this would look like a perfect result, but it is 
useless, because you would never have the answer already known before making a 
prediction. I removed this column and kept only the honest features.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Limitation:** Only about 37% of rows in this month (3,611,061 out of 9,841,378) 
have `gsc_data_available = TRUE`. This means most rows in the raw table do not have 
real search data, and were excluded before building features. My opportunity scores 
are only based on the pages and clients that had usable data this month — some 
clients or pages may be missing entirely if their tracking wasn't set up yet, so 
this slice does not represent every page in the warehouse.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.